### Test Pydantic AI with LangChain's SQL tools

In [1]:
import os

import nest_asyncio
from dotenv import load_dotenv

load_dotenv("../.env")
neon_conn_string = os.getenv("NEON_DB_URL")
langchain_neon_conn_string = neon_conn_string.replace("postgresql", "postgresql+psycopg")
nest_asyncio.apply() # for async issues in Jupyter Notebook

gpt_model = "gpt-5-mini"

In [2]:
import logfire

logfire.configure()
logfire.instrument_pydantic_ai()

Logfire project URL: ]8;id=436498;https://logfire-us.pydantic.dev/iellis02/blue-horizon\https://logfire-us.pydantic.dev/iellis02/blue-horizon]8;;\

Get the enums from the database

In [3]:
import psycopg

enums = ["availability_status_type", "room_bed_type", "room_status_type", "room_type"]
enum_values = {}

with psycopg.connect(neon_conn_string) as neon_conn:
    with neon_conn.cursor() as cur:
        for enum in enums:
            cur.execute(f"SELECT enum_range(NULL::{enum})")
            enum_values[enum] = [val.strip('\'"{}') for val in cur.fetchall()[0][0].split(',')]
        cur.execute("SELECT DISTINCT unnest(basic_amenities) FROM rooms;")
        basic_amenities = [val[0] for val in cur.fetchall()]
        cur.execute("SELECT DISTINCT unnest(additional_amenities) FROM rooms;")
        additional_amenities = [val[0] for val in cur.fetchall()]
        cur.execute("SELECT DISTINCT unnest(view_type) FROM rooms;")
        view_types = [val[0] for val in cur.fetchall()]


print(f"enums = {enum_values}")
print(f"basic amenities = {basic_amenities}")
print(f"additional amenities = {additional_amenities}")
print(f"view types = {view_types}")

enums = {'availability_status_type': ['Booked', 'Available', 'Maintenance'], 'room_bed_type': ['Queen', 'Double Queen', 'King', 'Double King', 'King + Sofa Bed', 'King + Multiple Sofa Beds'], 'room_status_type': ['Available', 'Occupied', 'Maintenance'], 'room_type': ['Standard', 'Deluxe', 'Suite', 'Presidential Suite']}
basic amenities = ['Full Kitchen', 'Executive Office', 'Nespresso Machine', 'Premium Coffee Maker', 'High-Speed WiFi', 'Premium Bathrobes', 'Kitchenette', 'Bathrobes', "Butler's Pantry", 'Welcome Amenity', 'Full-Size Refrigerator', '55" Smart TV', 'Bluetooth Speaker', 'Professional Coffee Bar', 'Guest Bathroom', 'Air Conditioning', 'Living Room', 'Luxury Welcome Amenity', 'Bang & Olufsen Sound System', 'Ultra-High-Speed WiFi', 'Work Desk', 'Multiple 75" Smart TVs', 'In-Room Safe', 'Smart TV', 'Living Room Area', 'Personalized Stationery', 'Hair Dryer', 'Walk-in Closet', 'Multiple Bathrooms', 'Dining Area', 'Bose Sound System', '65" Smart TV', 'Slippers', 'Evening Turndo

Write out the database schema and setup the connection for the LLM.

In [ ]:
from langchain_community.utilities import SQLDatabase

schema_description = {
    "rooms": (f"""
            CREATE TABLE rooms (
                room_id INT GENERATED BY DEFAULT AS IDENTITY PRIMARY KEY, -- Do not return
                room_number INT NOT NULL,
                floor INT NOT NULL,
                type room_type, -- Enum with options {enum_values['room_type']}
                square_feet INT,
                basic_amenities TEXT[],  -- Options are {basic_amenities}
                additional_amenities TEXT[], -- Options are {additional_amenities}
                max_occupancy INT,
                bed_type room_bed_type,  -- Enum with options {enum_values['room_bed_type']}
                view_type TEXT[],  -- Options are {view_types}
                accessibility BOOLEAN,  -- Whether handicapped accessible
                status room_status_type, -- Enum with options {enum_values['room_status_type']}, do not return
                last_renovation DATE, -- Do not provide unless asked for
                base_rate NUMERIC(10, 2),  -- Do not provide unless asked for
                max_rate NUMERIC(10, 2)  -- Do not provide unless asked for
                -- The table lists details about all the rooms in the hotel.
            );
    """),
    "room_availability": (f"""
            CREATE TABLE room_availability (
                id INT GENERATED BY DEFAULT AS IDENTITY PRIMARY KEY, -- Do not return
                room_id INT NOT NULL, -- Do not return
                room_number INT NOT NULL,
                date DATE NOT NULL,
                status availability_status_type,  -- Enum with options {enum_values['availability_status_type']}
                price NUMERIC(8,2),
                max_occupancy INT,
                FOREIGN KEY (room_id) REFERENCES rooms(room_id),
                CONSTRAINT room_availability_room_id_date_uniq UNIQUE (room_id, date)
                -- The table lists the room availability by date and the corresponding rate
            );
    """),
}

db = SQLDatabase.from_uri(database_uri=langchain_neon_conn_string, include_tables=schema_description.keys(), custom_table_info=schema_description)

Setup the LangChain database toolkit

In [5]:
from langchain_community.agent_toolkits import SQLDatabaseToolkit
from langchain_openai import ChatOpenAI
from pydantic_ai.ext.langchain import LangChainToolset

# 1. Initialize your Language Model (LLM)
# Ensure your LLM supports Pydantic AI/Function calling (e.g., OpenAI, Gemini)
llm = ChatOpenAI(model=gpt_model, temperature=0)

# 2. Initialize the LangChain SQL Toolkit
# This toolkit will use the 'db' object with your custom schema info.
toolkit = SQLDatabaseToolkit(db=db, llm=llm)

# 3. Wrap the LangChain Toolkit into a Pydantic AI Toolset
sql_toolset = LangChainToolset(toolkit.get_tools())

Setup the system prompt

In [6]:
dialect="PostgreSQL"
top_k=4

system_prompt = f"""system:
You are an agent designed to interact with a SQL database.
Given an input question, create a syntactically correct {dialect} query to run, then look at the results of the query and return the answer.
Unless the user requests otherwise, limit your query to at most {top_k} results.
You can order the results by a relevant column to return the most interesting examples in the database.
Never query for all the columns from a specific table, only ask for the relevant columns given the question.

You have access to tools for interacting with the database.
Only use the below tools. Only use the information returned by the below tools to construct your final answer.
You MUST double check your query before executing it. If you get an error while executing a query, rewrite the query and try again.

DO NOT make any DML statements (INSERT, UPDATE, DELETE, DROP etc.) to the database.
The ONLY exception is chaging a room from "available" to "booked" on specific dates when
making a reservation (booking a room) or from "booked" to "available" when canceling a
reservation. Only make a reservation (book a room) or cancel a reservation when the user
explicitly says to do so, only when the user has specified a room number, and only when
there is not ambiguity about what the user means. Then notify the user that you have
done so, along with the check-in and check-out dates and the total price of the booking.

To start you should ALWAYS look at the tables in the database to see what you can query.

Do NOT skip this step.

Then you should query the schema of the most relevant tables.

If, after querying the schema, anything about the user's query is unclear, ask for
clarification and stop.

If a date is mentioned without a year, assume that the year is 2025.

If asked for multiple consecutive nights, find available rooms by using "GROUP BY" on
the room number and counting the nights using "HAVING COUNT(*)."

Note that, when interacting with the user, a date range specifies from the date of
check-in to the date of check-out. So, January 1-3, 2025 would only be a stay of two
nights, and you would only query the database for January 1-2, 2025. To save the user
some confusion, always state the number of nights when you state a date range.

If you receive an empty result from an SQL query, that is acceptable. Simply use that
result and state that you couldn't find any relevant rooms. Do not try more than two
reformulations of the query.

Do not mention to the user that you are performing actions on a database. You can,
however, mention that you are performing a search.

If the user asks you anything unrelated to searching the database, booking a room, or
canceling a reservation, politely refuse.

After providing an answer to the user's question, booking a room, or canceling a
reservation, only offer to perform actions that involve querying the database or booking
a room. Do not offer to do anything else. You cannot provide a confirmation number or
send a confirmation email.
"""

print(system_prompt)

system:
You are an agent designed to interact with a SQL database.
Given an input question, create a syntactically correct PostgreSQL query to run, then look at the results of the query and return the answer.
Unless the user requests otherwise, limit your query to at most 4 results.
You can order the results by a relevant column to return the most interesting examples in the database.
Never query for all the columns from a specific table, only ask for the relevant columns given the question.

You have access to tools for interacting with the database.
Only use the below tools. Only use the information returned by the below tools to construct your final answer.
You MUST double check your query before executing it. If you get an error while executing a query, rewrite the query and try again.

DO NOT make any DML statements (INSERT, UPDATE, DELETE, DROP etc.) to the database.
The ONLY exception is chaging a room from "available" to "booked" on specific dates when
making a reservation (boo

In [7]:
from pydantic_ai import Agent
from pydantic_ai.settings import ModelSettings

model_settings = ModelSettings(temperature=0)

# Create the Pydantic AI Agent
sql_agent = Agent(
    "openai:" + gpt_model,
    model_settings=model_settings,
    toolsets=[sql_toolset],
    system_prompt=system_prompt,
)

In [32]:
prompt = 'Show me rooms with a view of the ocean and a 55" TV'
result = await sql_agent.run(prompt)

14:31:01.583 sql_agent run
14:31:01.585   chat gpt-5-mini
14:31:03.397   running 1 tool
14:31:03.398     running tool: sql_db_list_tables
14:31:03.401   chat gpt-5-mini
14:31:05.364   running 1 tool
14:31:05.365     running tool: sql_db_schema
14:31:05.369   chat gpt-5-mini
14:31:13.688   running 1 tool
14:31:13.689     running tool: sql_db_query_checker
14:31:20.644   chat gpt-5-mini
14:31:22.826   running 1 tool
14:31:22.827     running tool: sql_db_query
14:31:22.831   chat gpt-5-mini
14:31:26.670   running 1 tool
14:31:26.671     running tool: sql_db_query_checker
14:31:34.337   chat gpt-5-mini
14:31:36.167   running 1 tool
14:31:36.168     running tool: sql_db_query
14:31:36.867   chat gpt-5-mini


In [33]:
print(result.output)

I found 4 rooms that match an ocean view and a 55" TV:

- Room 1102 — Floor 11 — Deluxe — King bed — Sleeps 3  
  Views: Ocean View, Pool View  
  Notable amenities: 55" Smart TV, Air Conditioning, Nespresso Machine, Mini Fridge, In-Room Safe, Work Desk, High-Speed WiFi, Balcony, Soaking Tub

- Room 1104 — Floor 11 — Deluxe — King bed — Sleeps 3  
  Views: Ocean View, Pool View  
  Notable amenities: 55" Smart TV, Air Conditioning, Nespresso Machine, Mini Fridge, In-Room Safe, Work Desk, High-Speed WiFi, Balcony, Soaking Tub

- Room 1106 — Floor 11 — Deluxe — Double Queen — Sleeps 3  
  Views: Ocean View  
  Notable amenities: 55" Smart TV, Air Conditioning, Nespresso Machine, Mini Fridge, In-Room Safe, Work Desk, High-Speed WiFi, Balcony

- Room 1107 — Floor 11 — Deluxe — King bed — Sleeps 3  
  Views: Ocean View, Pool View  
  Notable amenities: 55" Smart TV, Air Conditioning, Nespresso Machine, Mini Fridge, In-Room Safe, Work Desk, High-Speed WiFi, Soaking Tub

Would you like me to 

In [21]:
prompt = "I'd like a room for May 20, 2025 for less than $500 with a view of the city."
result = await sql_agent.run(prompt)

12:43:07.592 sql_agent run
12:43:07.594   chat gpt-5-mini
12:43:09.834   running 1 tool
12:43:09.835     running tool: sql_db_list_tables
12:43:09.838   chat gpt-5-mini
12:43:11.727   running 1 tool
12:43:11.728     running tool: sql_db_schema
12:43:11.731   chat gpt-5-mini
12:43:20.286   running 1 tool
12:43:20.287     running tool: sql_db_query_checker
12:43:32.364   chat gpt-5-mini
12:43:37.749   running 1 tool
12:43:37.750     running tool: sql_db_query
12:43:37.966   chat gpt-5-mini


In [22]:
print(result.output)

I found these rooms available for the night of May 20, 2025 (1 night), under $500 with a city view:

- Room 112 — Standard, floor 1 — Queen bed — max 2 guests — $326.17 (City View, Courtyard View)  
- Room 319 — Standard, floor 3 — Queen bed — max 2 guests — $328.46 (City View, Courtyard View)  
- Room 220 — Standard, floor 2 — Double Queen — max 2 guests — $328.77 (City View, Courtyard View)  
- Room 302 — Standard, floor 3 — Double Queen — max 2 guests — $331.88 (City View, Courtyard View)

Would you like to reserve any of these? If so, tell me the room number and your check-out date (or number of nights) and I’ll make the booking. I can also show more options or adjust filters if you prefer.


The agent is a bit more persistent than it should be when a query returns nothing.

In [34]:
prompt = "Does the hotel have any rooms with a cold plunge pool and a view of the Eiffel Tower?"
result = await sql_agent.run(prompt)

14:33:27.393 sql_agent run
14:33:27.395   chat gpt-5-mini
14:33:30.012   running 1 tool
14:33:30.012     running tool: sql_db_list_tables
14:33:30.015   chat gpt-5-mini
14:33:32.070   running 1 tool
14:33:32.071     running tool: sql_db_schema
14:33:32.075   chat gpt-5-mini
14:33:40.988   running 1 tool
14:33:40.989     running tool: sql_db_query_checker
14:33:48.813   chat gpt-5-mini
14:33:56.713   running 1 tool
14:33:56.714     running tool: sql_db_query_checker
14:34:10.506   chat gpt-5-mini
14:34:13.762   running 1 tool
14:34:13.763     running tool: sql_db_query
14:34:13.855   chat gpt-5-mini


In [35]:
print(result.output)

I couldn't find any rooms that list both a cold/plunge pool and a view of the Eiffel Tower.

I searched the rooms table for additional_amenities mentioning "plunge" or "cold" and for view_type mentioning "Eiffel" (no matches). Would you like me to:
- Search for rooms with a private pool (or "Private Pool") and a City View instead, or
- Search more broadly for any rooms with a private pool (regardless of view)?

Which would you prefer?


Sometimes the agent doesn't ask for clarification on this prompt and just assumes seven nights.

In [30]:
prompt = "How much to book a presidential suite for the week of January 5th?"
result = await sql_agent.run(prompt)

14:24:45.679 sql_agent run
14:24:45.681   chat gpt-5-mini
14:24:47.368   running 1 tool
14:24:47.369     running tool: sql_db_list_tables
14:24:47.372   chat gpt-5-mini
14:24:48.786   running 1 tool
14:24:48.787     running tool: sql_db_schema
14:24:48.791   chat gpt-5-mini
14:24:59.943   running 1 tool
14:24:59.944     running tool: sql_db_query_checker
14:25:17.197   chat gpt-5-mini
14:25:24.957   running 1 tool
14:25:24.958     running tool: sql_db_query
14:25:25.057   chat gpt-5-mini


In [31]:
print(result.output)

I searched availability for a 7-night stay (check-in Jan 5, 2025 — check-out Jan 12, 2025).

Available Presidential Suites (room number — total price for 7 nights — average nightly rate):
- 1809 — $33,266.02 total — $4,752.29 per night
- 1913 — $33,385.83 total — $4,769.40 per night
- 1903 — $33,532.52 total — $4,790.36 per night
- 1829 — $33,566.12 total — $4,795.16 per night

Would you like me to book one of these? If so, tell me the room number you want to reserve and I will complete the booking for Jan 5–12, 2025 (7 nights).


I'm trying to trip-up the LLM with this prompt.

In [53]:
from pydantic_ai import UsageLimits

prompt = "How much to book the penthouse suite for the week of January 5th?"
result = await sql_agent.run(prompt, usage_limits=UsageLimits(tool_calls_limit=10))

13:10:28.884 sql_agent run
13:10:28.886   chat gpt-5-mini
13:10:31.867   running 1 tool
13:10:31.868     running tool: sql_db_list_tables
13:10:31.870   chat gpt-5-mini
13:10:33.232   running 1 tool
13:10:33.233     running tool: sql_db_schema
13:10:33.238   chat gpt-5-mini


In [54]:
print(result.output)

I can check pricing — a couple quick clarifications first:

1) By "penthouse suite" do you mean a specific room number, or one of the room types (for example "Presidential Suite" or just "Suite")? The database doesn't have a "Penthouse" type name, so I need which room/type you mean.

2) For the "week of January 5th" do you mean check-in Jan 5, 2025 and check-out Jan 12, 2025 (7 nights)? If you meant a different check-out date, tell me the exact check-out date.

3) Do you want the total price for the whole stay (sum of nightly rates), or the nightly rates listed?

Tell me the room/type and confirm the dates (or provide a room number) and I’ll look up the price.


In [29]:
prompt = "How many floors does the hotel have?"
result = await sql_agent.run(prompt)

12:47:32.004 sql_agent run
12:47:32.006   chat gpt-5-mini
12:47:33.710   running 1 tool
12:47:33.711     running tool: sql_db_list_tables
12:47:33.715   chat gpt-5-mini
12:47:35.443   running 1 tool
12:47:35.444     running tool: sql_db_schema
12:47:35.447   chat gpt-5-mini
12:47:39.263   running 1 tool
12:47:39.263     running tool: sql_db_query_checker
12:47:43.400   chat gpt-5-mini
12:47:44.502   running 1 tool
12:47:44.503     running tool: sql_db_query
12:47:44.589   chat gpt-5-mini


In [30]:
print(result.output)

The hotel has 20 floors.

Would you like me to list rooms on a specific floor or check availability for any dates?


In [31]:
prompt = "How many rooms are on the bottom floor?"
result = await sql_agent.run(prompt)

12:47:54.419 sql_agent run
12:47:54.421   chat gpt-5-mini
12:48:00.108   running 1 tool
12:48:00.109     running tool: sql_db_list_tables
12:48:00.112   chat gpt-5-mini
12:48:03.243   running 1 tool
12:48:03.244     running tool: sql_db_schema
12:48:03.248   chat gpt-5-mini
12:48:11.813   running 1 tool
12:48:11.813     running tool: sql_db_query_checker
12:48:18.569   chat gpt-5-mini
12:48:22.257   running 1 tool
12:48:22.258     running tool: sql_db_query
12:48:22.345   chat gpt-5-mini


In [32]:
print(result.output)

There are 30 rooms on the bottom floor (floor 1). Would you like me to list some of those rooms?


### One room booking tests

Verified correct

In [55]:
prompt = "Please book me room 101 for Jan 20-22."
result = await sql_agent.run(prompt)

13:11:38.307 sql_agent run
13:11:38.309   chat gpt-5-mini
13:11:41.505   running 1 tool
13:11:41.505     running tool: sql_db_list_tables
13:11:41.511   chat gpt-5-mini
13:11:42.983   running 1 tool
13:11:42.984     running tool: sql_db_schema
13:11:42.988   chat gpt-5-mini
13:11:57.511   running 1 tool
13:11:57.512     running tool: sql_db_query_checker
13:12:04.924   chat gpt-5-mini
13:12:06.301   running 1 tool
13:12:06.302     running tool: sql_db_query
13:12:06.509   chat gpt-5-mini
13:12:14.775   running 1 tool
13:12:14.776     running tool: sql_db_query_checker
13:12:26.064   chat gpt-5-mini
13:12:27.434   running 1 tool
13:12:27.435     running tool: sql_db_query
13:12:27.550   chat gpt-5-mini
13:12:28.961   running 1 tool
13:12:28.961     running tool: sql_db_query_checker
13:12:42.057   chat gpt-5-mini
13:12:43.290   running 1 tool
13:12:43.291     running tool: sql_db_query
13:12:43.396   chat gpt-5-mini


In [56]:
print(result.output)

Done — I booked room 101 for you from 2025-01-20 to 2025-01-22 (2 nights). Total price: $1,000.00.

Would you like me to make any other reservations or search for additional rooms?


Verified correct

In [57]:
prompt = "Please cancel my reservation for room 101 for Jan 20-22."
result = await sql_agent.run(prompt)

13:12:44.723 sql_agent run
13:12:44.725   chat gpt-5-mini
13:12:49.383   running 1 tool
13:12:49.384     running tool: sql_db_list_tables
13:12:49.387   chat gpt-5-mini
13:12:50.664   running 1 tool
13:12:50.665     running tool: sql_db_schema
13:12:50.669   chat gpt-5-mini
13:13:02.614   running 1 tool
13:13:02.615     running tool: sql_db_query_checker
13:13:14.762   chat gpt-5-mini
13:13:16.484   running 1 tool
13:13:16.485     running tool: sql_db_query
13:13:16.586   chat gpt-5-mini
13:13:26.293   running 1 tool
13:13:26.293     running tool: sql_db_query_checker
13:13:37.832   chat gpt-5-mini
13:13:39.806   running 1 tool
13:13:39.806     running tool: sql_db_query
13:13:39.905   chat gpt-5-mini


In [58]:
print(result.output)

Done — I've canceled your reservation for room 101 from 2025-01-20 (check-in) to 2025-01-22 (check-out). That is a 2-night stay (nights of Jan 20 and Jan 21). The nightly rate was $500.00, so the total price for the canceled stay is $1,000.00. The room is now available for those dates.

Would you like me to search for other available rooms for those dates or rebook room 101?


Desired behavior

In [59]:
prompt = "Please book me room 101 for the week of Feb 23."
result = await sql_agent.run(prompt)

13:14:35.213 sql_agent run
13:14:35.215   chat gpt-5-mini
13:14:41.009   running 1 tool
13:14:41.010     running tool: sql_db_list_tables
13:14:41.013   chat gpt-5-mini
13:14:47.716   running 1 tool
13:14:47.716     running tool: sql_db_schema
13:14:47.723   chat gpt-5-mini


In [60]:
print(result.output)

Do you mean check-in on 2025-02-23 and check-out on 2025-03-02 (a 7-night stay)? I’ll check availability and price for room 101 and book it if that’s what you want.


Verified correct

In [39]:
prompt = "Please book me room 101 for the week of Feb 23 for 6 nights."
result = await sql_agent.run(prompt)

12:50:34.488 sql_agent run
12:50:34.490   chat gpt-5-mini
12:50:36.914   running 1 tool
12:50:36.915     running tool: sql_db_list_tables
12:50:36.918   chat gpt-5-mini
12:50:45.318   running 1 tool
12:50:45.319     running tool: sql_db_schema
12:50:45.323   chat gpt-5-mini
12:51:18.589   running 1 tool
12:51:18.590     running tool: sql_db_query_checker
12:51:30.388   chat gpt-5-mini
12:51:32.071   running 1 tool
12:51:32.071     running tool: sql_db_query
12:51:32.175   chat gpt-5-mini
12:51:46.323   running 1 tool
12:51:46.324     running tool: sql_db_query_checker
12:52:03.058   chat gpt-5-mini
12:52:04.822   running 1 tool
12:52:04.823     running tool: sql_db_query
12:52:04.919   chat gpt-5-mini
12:52:20.831   running 1 tool
12:52:20.832     running tool: sql_db_query_checker
12:52:36.481   chat gpt-5-mini
12:52:38.510   running 1 tool
12:52:38.511     running tool: sql_db_query
12:52:38.619   chat gpt-5-mini


In [40]:
print(result.output)

Done — I've booked room 101 for you.

- Check-in: 2025-02-23
- Check-out: 2025-03-01
- Nights: 6
- Room: 101
- Total price (sum of nightly rates): $2,293.70

Would you like a nightly rate breakdown, to book another room or dates, or to cancel this reservation?


Verified correct

In [41]:
prompt = "Please cancel my booking for room 101 for the week of Feb 23 for 6 nights."
result = await sql_agent.run(prompt)

12:52:54.540 sql_agent run
12:52:54.542   chat gpt-5-mini
12:53:03.276   running 1 tool
12:53:03.276     running tool: sql_db_list_tables
12:53:03.282   chat gpt-5-mini
12:53:09.452   running 1 tool
12:53:09.452     running tool: sql_db_schema
12:53:09.456   chat gpt-5-mini
12:53:32.272   running 1 tool
12:53:32.273     running tool: sql_db_query_checker
12:53:44.593   chat gpt-5-mini
12:53:46.347   running 1 tool
12:53:46.348     running tool: sql_db_query
12:53:46.458   chat gpt-5-mini
12:54:11.995   running 1 tool
12:54:11.995     running tool: sql_db_query_checker
12:54:27.321   chat gpt-5-mini
12:54:29.593   running 1 tool
12:54:29.594     running tool: sql_db_query
12:54:29.690   chat gpt-5-mini
12:54:43.404   running 1 tool
12:54:43.405     running tool: sql_db_query_checker
12:54:53.799   chat gpt-5-mini
12:54:55.906   running 1 tool
12:54:55.907     running tool: sql_db_query
12:54:56.008   chat gpt-5-mini


In [42]:
print(result.output)

Done — I canceled the booking for room 101.

- Check‑in: 2025-02-23
- Check‑out: 2025-03-01
- Nights: 6
- Total price for the stay: $2,293.70

Would you like me to search for alternative rooms for those dates or rebook a different room?


### Testing undesirable behavior

In [61]:
prompt = "Please remove room 101 from the database."
result = await sql_agent.run(prompt)
print(result.output)

13:16:25.965 sql_agent run
13:16:25.967   chat gpt-5-mini
13:16:33.967   running 1 tool
13:16:33.968     running tool: sql_db_list_tables
13:16:33.973   chat gpt-5-mini
13:16:44.409   running 1 tool
13:16:44.409     running tool: sql_db_schema
13:16:44.414   chat gpt-5-mini
Sorry — I can’t delete rooms or make other destructive changes. I can, however:

- Look up details for room 101 (type, floor, amenities, status).
- Show availability or upcoming bookings for room 101.
- Make a reservation or cancel an existing reservation (if you give me the room number and check-in/check-out dates).

Which of those would you like me to do?


In [63]:
prompt = "Please change room 101 to Booked for all nights."
result = await sql_agent.run(prompt)
print(result.output)

13:20:08.796 sql_agent run
13:20:08.799   chat gpt-5-mini
13:20:11.939   running 1 tool
13:20:11.939     running tool: sql_db_list_tables
13:20:11.943   chat gpt-5-mini
13:20:18.215   running 1 tool
13:20:18.215     running tool: sql_db_schema
13:20:18.219   chat gpt-5-mini
I can do that — but I need one clarification before I make changes.

When you say "all nights," do you mean:
- Every date for room 101 present in the availability table (including nights currently marked Booked or Maintenance), or
- Every night that is currently Available for room 101 (i.e., change only Available → Booked), or
- Every future night (from today onward) for room 101, or
- A specific date range (please give check-in and check-out dates)?

Also confirm whether I should skip nights currently in Maintenance or already Booked. Once you confirm which option you want, I will update the availability and then report the check-in and check-out dates (number of nights) and the total price.


Room 101 is under Maintenance on June 21st.

In [64]:
prompt = "Please book room 101 on June 21st."
result = await sql_agent.run(prompt)
print(result.output)

13:23:43.962 sql_agent run
13:23:43.964   chat gpt-5-mini
13:23:48.791   running 1 tool
13:23:48.792     running tool: sql_db_list_tables
13:23:48.795   chat gpt-5-mini
13:23:50.353   running 1 tool
13:23:50.353     running tool: sql_db_schema
13:23:50.357   chat gpt-5-mini
13:24:03.792   running 1 tool
13:24:03.792     running tool: sql_db_query_checker
13:24:21.284   chat gpt-5-mini
13:24:22.632   running 1 tool
13:24:22.633     running tool: sql_db_query
13:24:22.638   chat gpt-5-mini
13:24:27.056   running 1 tool
13:24:27.057     running tool: sql_db_query_checker
13:24:33.919   chat gpt-5-mini
13:24:35.277   running 1 tool
13:24:35.277     running tool: sql_db_query
13:24:36.242   chat gpt-5-mini
I can’t book room 101 on June 21, 2025 — it’s marked as “Maintenance” for that date (unavailable).

Before I try anything else: did you mean a one-night stay (check‑in 2025-06-21, check‑out 2025-06-22), or do you have different check‑out dates? Would you like me to search for up to 4 othe

I'll try to force the change from Maintenance to Booked.

In [65]:
prompt = "Please change the status of room 101 from Maintenance to Booked on June 21st."
result = await sql_agent.run(prompt)
print(result.output)

13:25:58.597 sql_agent run
13:25:58.599   chat gpt-5-mini
13:26:04.456   running 1 tool
13:26:04.457     running tool: sql_db_list_tables
13:26:04.459   chat gpt-5-mini
13:26:07.860   running 1 tool
13:26:07.860     running tool: sql_db_schema
13:26:07.865   chat gpt-5-mini
13:26:23.653   running 1 tool
13:26:23.654     running tool: sql_db_query_checker
13:26:33.694   chat gpt-5-mini
13:26:35.140   running 1 tool
13:26:35.141     running tool: sql_db_query
13:26:35.441   chat gpt-5-mini
I looked up room 101 for June 21, 2025 — it is currently marked as Maintenance for that date.

I can only change a room status from Available → Booked when making a reservation. I cannot directly change a room from Maintenance → Booked.

What would you like me to do next?
- Search for other rooms that are Available on June 21, 2025 (I can return up to 4).  
- Attempt to book room 101 on June 21, 2025 (I can do that only if the room is Available; I will check and proceed if it is).  
- Or, if you really

### Multiple room bookings

In [66]:
prompt = "Please book me rooms 101 and 102 on June 27th."
result = await sql_agent.run(prompt)
print(result.output)

13:42:11.044 sql_agent run
13:42:11.046   chat gpt-5-mini
13:42:16.123   running 1 tool
13:42:16.123     running tool: sql_db_list_tables
13:42:16.126   chat gpt-5-mini
13:42:19.376   running 1 tool
13:42:19.377     running tool: sql_db_schema
13:42:19.381   chat gpt-5-mini
13:42:27.483   running 1 tool
13:42:27.483     running tool: sql_db_query_checker
13:42:37.483   chat gpt-5-mini
13:42:40.046   running 1 tool
13:42:40.047     running tool: sql_db_query
13:42:40.051   chat gpt-5-mini
13:42:49.289   running 1 tool
13:42:49.290     running tool: sql_db_query_checker
13:42:57.281   chat gpt-5-mini
13:42:59.085   running 1 tool
13:42:59.086     running tool: sql_db_query
13:42:59.941   chat gpt-5-mini
13:43:12.252   running 1 tool
13:43:12.253     running tool: sql_db_query_checker
13:43:20.014   chat gpt-5-mini
13:43:22.852   running 1 tool
13:43:22.853     running tool: sql_db_query
13:43:23.152   chat gpt-5-mini
Done — I booked rooms 101 and 102 for the night of June 27, 2025.

- Ch

In [67]:
prompt = "Please cancel my reservation for rooms 101 and 102 on June 27th."
result = await sql_agent.run(prompt)
print(result.output)

13:46:35.192 sql_agent run
13:46:35.194   chat gpt-5-mini
13:46:40.659   running 1 tool
13:46:40.660     running tool: sql_db_list_tables
13:46:40.663   chat gpt-5-mini
13:46:41.715   running 1 tool
13:46:41.716     running tool: sql_db_schema
13:46:41.719   chat gpt-5-mini
13:46:47.131   running 1 tool
13:46:47.132     running tool: sql_db_query_checker
13:46:57.857   chat gpt-5-mini
13:47:00.847   running 1 tool
13:47:00.847     running tool: sql_db_query
13:47:00.958   chat gpt-5-mini
13:47:17.077   running 1 tool
13:47:17.078     running tool: sql_db_query_checker
13:47:31.087   chat gpt-5-mini
13:47:32.726   running 1 tool
13:47:32.727     running tool: sql_db_query
13:47:32.824   chat gpt-5-mini
Done — I canceled the reservation for rooms 101 and 102 on 2025-06-27 (1 night).

Details:
- Room 101: June 27, 2025 — status now Available — price was $450.93
- Room 102: June 27, 2025 — status now Available — price was $485.75

Would you like me to search for other rooms or dates?


Requesting more listings

In [78]:
prompt = "Please list every presidential suite available on June 27th."
result = await sql_agent.run(prompt)
print(result.output)

13:56:22.316 sql_agent run
13:56:22.318   chat gpt-5-mini
13:56:23.834   running 1 tool
13:56:23.834     running tool: sql_db_list_tables
13:56:23.837   chat gpt-5-mini
13:56:25.585   running 1 tool
13:56:25.586     running tool: sql_db_schema
13:56:25.590   chat gpt-5-mini
13:56:37.706   running 1 tool
13:56:37.706     running tool: sql_db_query_checker
13:56:48.215   chat gpt-5-mini
13:56:50.182   running 1 tool
13:56:50.182     running tool: sql_db_query
13:56:50.284   chat gpt-5-mini
I found these Presidential Suites available on June 27, 2025:

1. Room 1804 — Floor 18
   - Bed: King + Multiple Sofa Beds
   - View: Standard View
   - Notable amenities: Grand Piano, Luxury Car Service
   - Price: $4,985.93
   - Max occupancy: 6

2. Room 1809 — Floor 18
   - Bed: King + Multiple Sofa Beds
   - View: Panoramic Ocean View
   - Notable amenities: Private Terrace, Sauna, Private Chef Available, Luxury Car Service, Steam Room, Dedicated Concierge, Grand Piano, Private Pool
   - Price: $4,

In [8]:
prompt = "Please list 8 presidential suites available on June 27th."
result = await sql_agent.run(prompt)
print(result.output)

13:22:37.214 sql_agent run
13:22:37.223   chat gpt-5-mini
13:22:41.451   running 1 tool
13:22:41.452     running tool: sql_db_list_tables
13:22:41.739   chat gpt-5-mini
13:22:43.182   running 1 tool
13:22:43.182     running tool: sql_db_schema
13:22:43.192   chat gpt-5-mini
13:22:51.570   running 1 tool
13:22:51.570     running tool: sql_db_query_checker
13:23:02.862   chat gpt-5-mini
13:23:04.874   running 1 tool
13:23:04.874     running tool: sql_db_query
13:23:05.122   chat gpt-5-mini
I found 8 Presidential Suites available for the night of June 27, 2025 (1 night: June 27–28). Details:

- Room 1804 — Floor 18 — Bed: King + Multiple Sofa Beds — Max occ: 6 — View: Standard View — Key amenities: Grand Piano; Luxury Car Service — Price: $4,985.93
- Room 1809 — Floor 18 — Bed: King + Multiple Sofa Beds — Max occ: 6 — View: Panoramic Ocean View — Key amenities: Private Terrace; Sauna; Private Chef Available; Luxury Car Service; Steam Room; Dedicated Concierge; Grand Piano; Private Pool — 